# Imports

In [1]:
import re
import torch
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from category_encoders import CatBoostEncoder

from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import TargetEncoder, StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold

from pytorch_tabnet.tab_model import TabNetClassifier

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

In [4]:
column_transformer = ColumnTransformer([
    (
        'target_encoder', 
        TargetEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'catboost_encoder', 
        CatBoostEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'standard_scaler', 
        StandardScaler(), 
        ['lapnumber', 'position', 'raceprogress', 'year', 'position_norm', 'race_progress_sin', 'position_vs_mean']
    ),
    (
        'robust_scaler', 
        RobustScaler(), 
        [
            'position_change', 'cumulative_degradation', 'laptime_delta', 'laptime_s', 'stint', 'driver_mean_lap', 'tyrelife', 'delta_x_tyre_life', 
            'compound_tyre_life', 'stint_progress', 'tyre_life_ratio', 'degradation_per_lap', 'position_change_cum', 'laps_since_pit', 'lap_time_inv',  
            'lap_time_vs_race_mean', 'lap_time_x_tyre', 'position_x_progress', 'degradation_x_progress', 'race_progress_squared', 'driver_avg_position' 
        ]
    ),
], remainder="passthrough")

# Loading Dataset

In [5]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [6]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Machine Learning

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):

    print(f"\n======== Fold {fold + 1} ========")

    X_tr = X_train.iloc[train_idx, :]
    X_val = X_train.iloc[valid_idx, :]

    y_tr = y_train.iloc[train_idx, 0]
    y_val = y_train.iloc[valid_idx, 0]

    X_tr_transformed = column_transformer.fit_transform(X_tr, y_tr)
    X_val_transformed = column_transformer.transform(X_val)

    tab_net = TabNetClassifier(
        n_d=32,
        n_a=32,
        n_steps=5,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-2),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={"step_size": 20, "gamma": 0.9},
        mask_type="entmax",
        seed=42,
        verbose=1
    )

    tab_net.fit(
        X_train=X_tr_transformed,
        y_train=y_tr,
        eval_set=[(X_val_transformed, y_val)],
        eval_name=["valid"],
        eval_metric=["auc"],
        max_epochs=200,
        patience=20,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=11,
        drop_last=False
    )

    score = tab_net.predict_proba(X_val_transformed)[:, 1]

    auc = roc_auc_score(y_val, score)
    aucs.append(auc)

    print(f"Fold AUC: {auc:.6f}")


print("\n==============================")
print(f"CV AUC: {np.mean(aucs):.6f}")
print(f"STD: {np.std(aucs):.6f}")
print("==============================")


======== Fold 1 ========


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.39905 | valid_auc: 0.88501 |  0:00:18s
epoch 1  | loss: 0.30953 | valid_auc: 0.90682 |  0:00:35s
epoch 2  | loss: 0.29449 | valid_auc: 0.91439 |  0:00:53s
epoch 3  | loss: 0.28437 | valid_auc: 0.92101 |  0:01:10s
epoch 4  | loss: 0.28378 | valid_auc: 0.91661 |  0:01:28s
epoch 5  | loss: 0.2822  | valid_auc: 0.923   |  0:01:45s
epoch 6  | loss: 0.27348 | valid_auc: 0.92585 |  0:02:03s
epoch 7  | loss: 0.26977 | valid_auc: 0.92628 |  0:02:20s
epoch 8  | loss: 0.26938 | valid_auc: 0.92656 |  0:02:38s
epoch 9  | loss: 0.26926 | valid_auc: 0.92697 |  0:02:56s
epoch 10 | loss: 0.26968 | valid_auc: 0.92487 |  0:03:13s
epoch 11 | loss: 0.26685 | valid_auc: 0.9282  |  0:03:31s
epoch 12 | loss: 0.26907 | valid_auc: 0.925   |  0:03:48s
epoch 13 | loss: 0.27119 | valid_auc: 0.91891 |  0:04:06s
epoch 14 | loss: 0.28187 | valid_auc: 0.92133 |  0:04:23s
epoch 15 | loss: 0.27755 | valid_auc: 0.92453 |  0:04:41s
epoch 16 | loss: 0.27201 | valid_auc: 0.92644 |  0:04:59s
epoch 17 | los

KeyboardInterrupt: 

# Deploy

In [ ]:
dump_pickle(voting, '../models/tabnet_column_transformer.pkl')

In [ ]:
tab_net.save_model("../models/tabnet_model")